In [1]:
from python_utilities.db_connection import DbConnection
import boto3
import json
import os
import pandas as pd
import ast

analytics_db = DbConnection('ANALYTICS', 'PROD_RDS')
session = boto3.Session(profile_name='739275445236_DataScienceUser')
s3 = session.client('s3')

INFO [2026-07-06 15:30:24] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file
INFO [2026-07-06 15:30:25] - Found credentials in shared credentials file: ~/.aws/credentials


In [15]:
# egvp intents egvp_id = None -> zendesk-letter and zendesk-email data

temp start

In [2]:
from python_utilities.db_connection import DbConnection
import boto3
import json
import os
import pandas as pd
import ast

analytics_db = DbConnection('ANALYTICS', 'PROD_RDS')
session = boto3.Session(profile_name='739275445236_DataScienceUser')
s3 = session.client('s3')

INFO [2026-07-06 15:30:27] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file
INFO [2026-07-06 15:30:27] - Found credentials in shared credentials file: ~/.aws/credentials


In [10]:
query = """
SELECT *
FROM llm_attachments_predictions lap
WHERE lap.created_at >= '2026-07-03'
AND (
    (type = 'drittauskunft_egvp' AND subtype='is_dritt' AND value = "'True'")
    OR (type = 'vermogenverzeichnis_egvp' AND subtype='is_va' AND value = "'True'")
    OR (type = 'egvp_standalone_invoice' AND value = "'True'")
)
"""
data = analytics_db.sql_to_df(query)


In [11]:
data

,id,created_at,model_name,type,subtype,value,attachment_id
0,12167952,2026-07-03 00:53:50,egvp_standalone_invoice,egvp_standalone_invoice,invoice,'True',70758299
1,12177274,2026-07-03 05:59:10,drittauskunft_egvp,drittauskunft_egvp,is_dritt,'True',70796006
2,12177350,2026-07-03 05:59:15,egvp_standalone_invoice,egvp_standalone_invoice,invoice,'True',70796007
3,12182805,2026-07-03 06:42:38,drittauskunft_egvp,drittauskunft_egvp,is_dritt,'True',70798131
4,12182860,2026-07-03 06:42:42,egvp_standalone_invoice,egvp_standalone_invoice,invoice,'True',70798130
...,...,...,...,...,...,...,...
252,12342908,2026-07-06 13:28:57,drittauskunft_egvp,drittauskunft_egvp,is_dritt,'True',71145614
253,12342930,2026-07-06 13:28:58,drittauskunft_egvp,drittauskunft_egvp,is_dritt,'True',71145645
254,12342998,2026-07-06 13:29:02,vermogenverzeichnis_egvp,vermogenverzeichnis_egvp,is_va,'True',71145571
255,12343119,2026-07-06 13:29:10,drittauskunft_egvp,drittauskunft_egvp,is_dritt,'True',71145613


In [26]:
import sys
sys.path.append('/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation')
from utils.prod_utils import pivot_attachment_predictions, get_data_by_attachment_id

In [14]:
pred = pivot_attachment_predictions(data)

In [15]:
pred

,attachment_id,drittauskunft_egvp_is_dritt,egvp_standalone_invoice_invoice,vermogenverzeichnis_egvp_is_va
0,70758299,none,True,none
1,70796006,True,none,none
2,70796007,none,True,none
3,70798130,none,True,none
4,70798131,True,none,none
...,...,...,...,...
245,71145644,none,True,none
246,71145645,True,none,none
247,71145657,none,True,none
248,71145658,True,none,none


In [24]:
cols = ['drittauskunft_egvp_is_dritt', 'vermogenverzeichnis_egvp_is_va', 'egvp_standalone_invoice_invoice']

send_dfs = []
for col in cols:
    subset = pred[pred[col] == 'True']
    subset_sample = subset.sample(n=15, random_state=42)
    send_dfs.append(subset_sample)
    
final_df = pd.concat(send_dfs)

In [ ]:
from python_utilities.db_connection import DbConnection
import boto3
analytics_db = DbConnection('ANALYTICS', 'PROD_RDS')
# Create session with specific profile
session = boto3.Session(profile_name='739275445236_DataScienceUser')
s3 = session.client('s3')

In [27]:
download_dir = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/send"

In [33]:
for df_name, df in zip(cols, send_dfs):
    for attachment_id in df['attachment_id']:
        download_full_path = os.path.join(download_dir, df_name)
        os.makedirs(download_full_path, exist_ok=True)
        get_data_by_attachment_id(attachment_id, analytics_db, s3, pdf_download=True, pdf_download_dir=download_full_path)


📎 ATTACHMENT SUMMARY
Attachment ID:   71021096
File Name:       71021096_Dokumente_58326_04072026_183056-signed.pdf

│ 📄 Document S3:
│    s3://pair-data-engineering-new/ocr_source_files/2026-07-04/egvp_id_404181/71021096_Dokumente_58326_04072026_183056-signed.pdf
│
│ 📝 Textract Output:
│    s3://pair-data-engineering-new/ocr_prepared_output/92183cad7d376678e77118d16990ecc844d632f9d81c2fd7ff5b2eab027975b0.json
│
│ 📊 Text Length:  3890 characters
│
│ 📥 PDF Downloaded: /Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/send/drittauskunft_egvp_is_dritt/71021096.pdf
│
│ 🤖 PREDICTIONS:
│    ⚠️  NOT LADUNG OR PFUB! 
│    Prob Ladung:              '0.08'
│    Prob Pfub:                '0.03'
│    Prob Vermogenverzeichnis: N/A
│    — invoice_detection_egvp —
│    Start Page:        '1'
│    End Page:          '2'
│    Is Invoice Inside: 'True'
│    — pfub_erlass_egvp —
│    Is Pfub:           'False'
│    Is Invoice Inside: 'True'
└────────────────────────────────────

temp end

In [16]:
query = """
SELECT *
FROM egvp_intents ei
WHERE ei.egvp_id IS NULL
"""
data = analytics_db.sql_to_df(query)

In [18]:
data.shape

(4826, 11)

In [23]:
print(f"Date between : {data['created_at'].min().date()} - {data['created_at'].max().date()}")

Date between : 2026-06-15 - 2026-07-06


In [26]:
data.ready_for_automation.value_counts()

ready_for_automation
0    4696
1     130
Name: count, dtype: int64

In [28]:
rfa = data[data.ready_for_automation == True]
rfa

,id,ticket_uuid,attachment_id,egvp_id,created_at,intents,ready_for_automation,status,retry,target_type,updated_at
0,39470,27432456-0da5-5c5e-8296-5fa101583f4e,28171800296348-1,None,2026-06-15 12:38:22,"[{""params"": {""slug"": ""196302948324"", ""debtor_n...",1,sent,0,egvp,2026-06-15 12:38:22
1,39472,208d0153-2109-56ed-8bd6-a261d66c494e,28172046035356-1,None,2026-06-15 14:08:45,"[{""params"": {""slug"": ""192298781407"", ""debtor_n...",1,sent,0,egvp,2026-06-15 14:08:45
2,39473,fe1d5a63-52e9-5802-a4d9-9f6a5411ad0c,28172638247964-1,None,2026-06-15 14:09:05,"[{""params"": {""slug"": ""177394173211"", ""debtor_n...",1,sent,0,egvp,2026-06-15 14:09:05
3,39474,50e1fc04-9265-55ac-b909-d14ba17d9fe4,28172903254428-1,None,2026-06-15 14:09:39,"[{""params"": {""slug"": ""127103324753"", ""debtor_n...",1,sent,0,egvp,2026-06-15 14:09:39
4,40116,59b85aa0-8473-56a1-b90f-08307764baa2,28200468095516-1,None,2026-06-16 09:59:41,"[{""params"": {""slug"": ""181396289017"", ""debtor_n...",1,sent,0,egvp,2026-06-16 09:59:41
...,...,...,...,...,...,...,...,...,...,...,...
3858,52413,65517563-a47d-5b54-8493-ce3ccb879a41,28649994484252-1,None,2026-07-03 17:22:01,"[{""params"": {""slug"": ""198384884153"", ""debtor_n...",1,sent,0,egvp,2026-07-03 17:22:01
3859,52414,c5f054b4-88e0-56ae-a68f-2510f85ef06e,28650226046876-1,None,2026-07-03 17:22:01,"[{""params"": {""slug"": ""171923686264"", ""debtor_n...",1,sent,0,egvp,2026-07-03 17:22:01
3860,52415,7637653a-a869-5856-9f14-f995daa5578b,28649889336348-1,None,2026-07-03 17:22:08,"[{""params"": {""slug"": ""142339270670"", ""debtor_n...",1,sent,0,egvp,2026-07-03 17:22:08
3861,52416,5065e5df-e9cc-5663-8c10-4d138ed2df3f,28649972137116-1,None,2026-07-03 17:22:11,"[{""params"": {""slug"": ""165653326372"", ""debtor_n...",1,sent,0,egvp,2026-07-03 17:22:11


In [37]:
import json
rfa['intents'] = rfa['intents'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

/var/folders/ym/hcyz4chn3cq4n_8dslg5k50h0000gn/T/ipykernel_39591/747993890.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rfa['intents'] = rfa['intents'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)


In [40]:
rfa['intents'].isnull().sum()

np.int64(0)

In [43]:
lengths = rfa['intents'].apply(lambda x: len(x) if isinstance(x, list) else 0)
lengths.value_counts()

intents
1    130
Name: count, dtype: int64

In [44]:
rfa['channel'] = rfa['intents'].apply(lambda x: x[0]['channel'] if isinstance(x, list) and len(x) > 0 and 'channel' in x[0] else None)

/var/folders/ym/hcyz4chn3cq4n_8dslg5k50h0000gn/T/ipykernel_39591/1689248179.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rfa['channel'] = rfa['intents'].apply(lambda x: x[0]['channel'] if isinstance(x, list) and len(x) > 0 and 'channel' in x[0] else None)


In [45]:
rfa['channel'].value_counts()

channel
zendesk           89
zendesk-email     39
zendesk-letter     2
Name: count, dtype: int64

In [46]:
rfa['channel'] = rfa['channel'].apply(lambda x: 'zendesk-email' if x == 'zendesk' else x)

/var/folders/ym/hcyz4chn3cq4n_8dslg5k50h0000gn/T/ipykernel_39591/864601171.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rfa['channel'] = rfa['channel'].apply(lambda x: 'zendesk-email' if x == 'zendesk' else x)


In [47]:
rfa['channel'].value_counts()

channel
zendesk-email     128
zendesk-letter      2
Name: count, dtype: int64

In [49]:
rfa

,id,ticket_uuid,attachment_id,egvp_id,created_at,intents,ready_for_automation,status,retry,target_type,updated_at,channel
0,39470,27432456-0da5-5c5e-8296-5fa101583f4e,28171800296348-1,None,2026-06-15 12:38:22,"[{'params': {'slug': '196302948324', 'debtor_n...",1,sent,0,egvp,2026-06-15 12:38:22,zendesk-email
1,39472,208d0153-2109-56ed-8bd6-a261d66c494e,28172046035356-1,None,2026-06-15 14:08:45,"[{'params': {'slug': '192298781407', 'debtor_n...",1,sent,0,egvp,2026-06-15 14:08:45,zendesk-email
2,39473,fe1d5a63-52e9-5802-a4d9-9f6a5411ad0c,28172638247964-1,None,2026-06-15 14:09:05,"[{'params': {'slug': '177394173211', 'debtor_n...",1,sent,0,egvp,2026-06-15 14:09:05,zendesk-email
3,39474,50e1fc04-9265-55ac-b909-d14ba17d9fe4,28172903254428-1,None,2026-06-15 14:09:39,"[{'params': {'slug': '127103324753', 'debtor_n...",1,sent,0,egvp,2026-06-15 14:09:39,zendesk-email
4,40116,59b85aa0-8473-56a1-b90f-08307764baa2,28200468095516-1,None,2026-06-16 09:59:41,"[{'params': {'slug': '181396289017', 'debtor_n...",1,sent,0,egvp,2026-06-16 09:59:41,zendesk-email
...,...,...,...,...,...,...,...,...,...,...,...,...
3858,52413,65517563-a47d-5b54-8493-ce3ccb879a41,28649994484252-1,None,2026-07-03 17:22:01,"[{'params': {'slug': '198384884153', 'debtor_n...",1,sent,0,egvp,2026-07-03 17:22:01,zendesk-email
3859,52414,c5f054b4-88e0-56ae-a68f-2510f85ef06e,28650226046876-1,None,2026-07-03 17:22:01,"[{'params': {'slug': '171923686264', 'debtor_n...",1,sent,0,egvp,2026-07-03 17:22:01,zendesk-email
3860,52415,7637653a-a869-5856-9f14-f995daa5578b,28649889336348-1,None,2026-07-03 17:22:08,"[{'params': {'slug': '142339270670', 'debtor_n...",1,sent,0,egvp,2026-07-03 17:22:08,zendesk-email
3861,52416,5065e5df-e9cc-5663-8c10-4d138ed2df3f,28649972137116-1,None,2026-07-03 17:22:11,"[{'params': {'slug': '165653326372', 'debtor_n...",1,sent,0,egvp,2026-07-03 17:22:11,zendesk-email


In [57]:
import copy
def find_how_automated(row):
    intent = copy.deepcopy(row['intents'][0])
    intent = str(intent)  # Convert intent to string if it's not already
    if intent:
        try:
            intent = intent.lower()
        except:
            return "none"
        if "ladung_va" in intent:
            return "ladung_va"
        elif "pfub_erlass" in intent:
            return "pfub_erlass"
        elif "vermogen" in intent:
            return "va"
        elif "dritt" in intent:
            return "dritt"
        elif '"aftercourt_type": "invoice"' in intent:
            return "invoice"
        else:
            return "none"
    else:
        return "none"

In [58]:
rfa['auto_type'] = rfa.apply(find_how_automated, axis=1)

/var/folders/ym/hcyz4chn3cq4n_8dslg5k50h0000gn/T/ipykernel_39591/1462749539.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rfa['auto_type'] = rfa.apply(find_how_automated, axis=1)


In [61]:
rfa['auto_type'].value_counts()

auto_type
ladung_va    130
Name: count, dtype: int64

In [63]:
rfa.groupby(['channel', 'auto_type']).size().reset_index(name='counts')

,channel,auto_type,counts
0,zendesk-email,ladung_va,128
1,zendesk-letter,ladung_va,2


Result: We only automate for ladung va -> Why this is happening?

Check-> Not automated data

In [64]:
no_auto = data[data.ready_for_automation == False]

In [65]:
no_auto

,id,ticket_uuid,attachment_id,egvp_id,created_at,intents,ready_for_automation,status,retry,target_type,updated_at
66,44469,c0cb6821-217b-5af3-9b40-7e4fdc2fec76,28421403923100-1,None,2026-06-24 18:32:44,"[{""params"": {""debtor_name"": ""Nanette Araneta"",...",0,sent,0,egvp,2026-06-24 18:32:44
73,45328,dbd25f59-a372-5c7b-a4fa-d4dd7bcb2e51,28468000637212-1,None,2026-06-26 13:12:03,"[{""params"": {""slug"": ""178921897074"", ""judicial...",0,sent,0,egvp,2026-06-26 13:12:03
76,45424,388417c6-a71d-51fc-b2fa-749282d1cd8e,28472281962780-1,None,2026-06-26 14:57:09,"[{""params"": {""debtor_name"": ""Yvonne Langosch"",...",0,sent,0,egvp,2026-06-26 14:57:09
98,46324,1064ad3f-edc5-5f56-b9f3-d3e3a3f6bba8,28555789622940-1,None,2026-06-30 12:53:15,"[{""params"": {}, ""channel"": ""zendesk-letter"", ""...",0,sent,0,egvp,2026-06-30 12:53:15
99,46325,3aa5cc8c-15f8-5efd-a38d-ddd88a29726e,28555867886108-1,None,2026-06-30 12:53:27,"[{""params"": {}, ""channel"": ""zendesk-letter"", ""...",0,sent,0,egvp,2026-06-30 12:53:27
...,...,...,...,...,...,...,...,...,...,...,...
4821,53594,e60533f2-53ef-559e-ae47-5b2cca079b6e,28695225004956-1,None,2026-07-06 08:32:48,"[{""params"": {}, ""channel"": ""zendesk-letter"", ""...",0,sent,0,egvp,2026-07-06 08:32:48
4822,53595,b6d87e6e-c62f-5429-ab45-ff74ddff950d,28695256844572-1,None,2026-07-06 08:32:49,"[{""params"": {}, ""channel"": ""zendesk-letter"", ""...",0,sent,0,egvp,2026-07-06 08:32:49
4823,53596,9ac66d1e-633a-590d-8a2d-77c9f6cef9dc,28695249161628-1,None,2026-07-06 08:32:50,"[{""params"": {}, ""channel"": ""zendesk-letter"", ""...",0,sent,0,egvp,2026-07-06 08:32:50
4824,53597,290329c7-8616-5dc8-a44a-18569000a55b,28695899100572-1,None,2026-07-06 08:33:36,"[{""params"": {}, ""channel"": ""zendesk-letter"", ""...",0,sent,0,egvp,2026-07-06 08:33:36


In [67]:
import json
no_auto['intents'] = no_auto['intents'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

/var/folders/ym/hcyz4chn3cq4n_8dslg5k50h0000gn/T/ipykernel_39591/2492881763.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  no_auto['intents'] = no_auto['intents'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)


In [69]:
no_auto['channel'] = no_auto['intents'].apply(lambda x: x[0]['channel'] if isinstance(x, list) and len(x) > 0 and 'channel' in x[0] else None)

/var/folders/ym/hcyz4chn3cq4n_8dslg5k50h0000gn/T/ipykernel_39591/4136349704.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  no_auto['channel'] = no_auto['intents'].apply(lambda x: x[0]['channel'] if isinstance(x, list) and len(x) > 0 and 'channel' in x[0] else None)


In [70]:
no_auto['channel'].value_counts()

channel
zendesk-letter    4693
zendesk              3
Name: count, dtype: int64

zendesk-email is only auotomated for ladung -> Why it would be: because of business logic.
Most of the not automated data is zendesk letter

In [72]:
zendesk_email_no_auto = no_auto[no_auto['channel'] == 'zendesk']
zendesk_email_no_auto

,id,ticket_uuid,attachment_id,egvp_id,created_at,intents,ready_for_automation,status,retry,target_type,updated_at,channel
66,44469,c0cb6821-217b-5af3-9b40-7e4fdc2fec76,28421403923100-1,None,2026-06-24 18:32:44,"[{'params': {'debtor_name': 'Nanette Araneta',...",0,sent,0,egvp,2026-06-24 18:32:44,zendesk
73,45328,dbd25f59-a372-5c7b-a4fa-d4dd7bcb2e51,28468000637212-1,None,2026-06-26 13:12:03,"[{'params': {'slug': '178921897074', 'judicial...",0,sent,0,egvp,2026-06-26 13:12:03,zendesk
76,45424,388417c6-a71d-51fc-b2fa-749282d1cd8e,28472281962780-1,None,2026-06-26 14:57:09,"[{'params': {'debtor_name': 'Yvonne Langosch',...",0,sent,0,egvp,2026-06-26 14:57:09,zendesk


3 data from zendesk email is not auto because 1 param is missing

## Check for zendesk letters

In [75]:
no_auto_letters = no_auto[no_auto['channel'] == 'zendesk-letter']

In [76]:
no_auto_letters

,id,ticket_uuid,attachment_id,egvp_id,created_at,intents,ready_for_automation,status,retry,target_type,updated_at,channel
98,46324,1064ad3f-edc5-5f56-b9f3-d3e3a3f6bba8,28555789622940-1,None,2026-06-30 12:53:15,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-06-30 12:53:15,zendesk-letter
99,46325,3aa5cc8c-15f8-5efd-a38d-ddd88a29726e,28555867886108-1,None,2026-06-30 12:53:27,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-06-30 12:53:27,zendesk-letter
100,46326,f7daf19f-d338-5cd6-bcc3-3756b33ec802,28555918825244-1,None,2026-06-30 12:53:28,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-06-30 12:53:28,zendesk-letter
101,46327,0275d36f-ea43-5737-ae25-524e8abea847,28556906987932-1,None,2026-06-30 12:53:38,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-06-30 12:53:38,zendesk-letter
102,46328,b919a767-a79f-5ad1-90e9-63739257ce24,28557101335836-1,None,2026-06-30 12:53:39,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-06-30 12:53:39,zendesk-letter
...,...,...,...,...,...,...,...,...,...,...,...,...
4821,53594,e60533f2-53ef-559e-ae47-5b2cca079b6e,28695225004956-1,None,2026-07-06 08:32:48,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-07-06 08:32:48,zendesk-letter
4822,53595,b6d87e6e-c62f-5429-ab45-ff74ddff950d,28695256844572-1,None,2026-07-06 08:32:49,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-07-06 08:32:49,zendesk-letter
4823,53596,9ac66d1e-633a-590d-8a2d-77c9f6cef9dc,28695249161628-1,None,2026-07-06 08:32:50,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-07-06 08:32:50,zendesk-letter
4824,53597,290329c7-8616-5dc8-a44a-18569000a55b,28695899100572-1,None,2026-07-06 08:33:36,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-07-06 08:33:36,zendesk-letter


look for detection

In [77]:
find_how_detected = find_how_automated  # Reusing the same function for detection

no_auto_letters['detected'] = no_auto_letters.apply(find_how_detected, axis=1)

/var/folders/ym/hcyz4chn3cq4n_8dslg5k50h0000gn/T/ipykernel_39591/2540839624.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  no_auto_letters['detected'] = no_auto_letters.apply(find_how_detected, axis=1)


In [78]:
no_auto_letters['detected'].value_counts()

detected
none           4689
pfub_erlass       2
ladung_va         2
Name: count, dtype: int64

In [79]:
ladung_letter_no_auto = no_auto_letters[no_auto_letters['detected'] == 'ladung_va']
pfub_letter_no_auto = no_auto_letters[no_auto_letters['detected'] == 'pfub_erlass']

In [81]:
ladung_letter_no_auto 
# Result: slug is missing, thats why not automated

,id,ticket_uuid,attachment_id,egvp_id,created_at,intents,ready_for_automation,status,retry,target_type,updated_at,channel,detected
1331,48093,50b18d34-fa84-5258-a1aa-496e5a12c404,28576736949276-1,None,2026-07-01 06:57:44,[{'params': {'debtor_name': 'Nikola Bohdansky'...,0,sent,0,egvp,2026-07-01 06:57:44,zendesk-letter,ladung_va
1337,48099,8487151f-c776-5a52-afdd-7b2783ebf776,28576721480220-1,None,2026-07-01 06:57:49,"[{'params': {'debtor_name': 'Fatima Frade', 'j...",0,sent,0,egvp,2026-07-01 06:57:49,zendesk-letter,ladung_va


In [82]:
pfub_letter_no_auto
# Result: slug is missing, thats why not automated

,id,ticket_uuid,attachment_id,egvp_id,created_at,intents,ready_for_automation,status,retry,target_type,updated_at,channel,detected
1173,47935,4ca67f10-bbf3-56cc-8a4b-8a98669af378,28576652321564-1,None,2026-07-01 06:53:35,"[{'params': {'debtor_name': 'Softic', 'credito...",0,sent,0,egvp,2026-07-01 06:53:35,zendesk-letter,pfub_erlass
4765,53536,9ba5e0e7-757b-5b7e-8f77-d7b8a5c33054,28694525463452-1,None,2026-07-06 07:21:33,"[{'params': {'slug': '163055985135', 'debtor_n...",0,sent,0,egvp,2026-07-06 07:21:33,zendesk-letter,pfub_erlass


In [84]:
# TODO: are you sure for zendesk-letters and zendesk-emails egvp models are running? 
# Looks like only zendesk pipeline is running, resulting no egvp outputs

In [85]:
no_auto_letters

,id,ticket_uuid,attachment_id,egvp_id,created_at,intents,ready_for_automation,status,retry,target_type,updated_at,channel,detected
98,46324,1064ad3f-edc5-5f56-b9f3-d3e3a3f6bba8,28555789622940-1,None,2026-06-30 12:53:15,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-06-30 12:53:15,zendesk-letter,none
99,46325,3aa5cc8c-15f8-5efd-a38d-ddd88a29726e,28555867886108-1,None,2026-06-30 12:53:27,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-06-30 12:53:27,zendesk-letter,none
100,46326,f7daf19f-d338-5cd6-bcc3-3756b33ec802,28555918825244-1,None,2026-06-30 12:53:28,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-06-30 12:53:28,zendesk-letter,none
101,46327,0275d36f-ea43-5737-ae25-524e8abea847,28556906987932-1,None,2026-06-30 12:53:38,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-06-30 12:53:38,zendesk-letter,none
102,46328,b919a767-a79f-5ad1-90e9-63739257ce24,28557101335836-1,None,2026-06-30 12:53:39,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-06-30 12:53:39,zendesk-letter,none
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4821,53594,e60533f2-53ef-559e-ae47-5b2cca079b6e,28695225004956-1,None,2026-07-06 08:32:48,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-07-06 08:32:48,zendesk-letter,none
4822,53595,b6d87e6e-c62f-5429-ab45-ff74ddff950d,28695256844572-1,None,2026-07-06 08:32:49,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-07-06 08:32:49,zendesk-letter,none
4823,53596,9ac66d1e-633a-590d-8a2d-77c9f6cef9dc,28695249161628-1,None,2026-07-06 08:32:50,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-07-06 08:32:50,zendesk-letter,none
4824,53597,290329c7-8616-5dc8-a44a-18569000a55b,28695899100572-1,None,2026-07-06 08:33:36,"[{'params': {}, 'channel': 'zendesk-letter', '...",0,sent,0,egvp,2026-07-06 08:33:36,zendesk-letter,none
